In [75]:
from langchain_core.tools import tool, InjectedToolArg
import requests
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage
from typing import Annotated

# Tool Creation

In [76]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """This function fetches the currect conversion rate between base currency and target currency."""
    url=f'https://v6.exchangerate-api.com/v6/5e3f623910d40d9730bc39d5/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()
@tool
def currency_converter(base_currency_value:float,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """This function convert the base currency into target currency with the help of given conversion rate"""
    return base_currency_value*conversion_rate

# Tool Binding

In [77]:
llm=ChatOpenAI()
llm_with_tools=llm.bind_tools([get_conversion_factor,currency_converter])

# Tool Execution

In [78]:
messages=[HumanMessage(content="What is conversion factor between USD and PKR. Also convert 10 USD to PKR")]

In [79]:
AImessage=llm_with_tools.invoke(messages)

In [80]:
AImessage

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QyRgXfRPU4lGOgN5Of6haBEX', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "PKR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_K0D9xLIcfYpx8gHjtija1fc1', 'function': {'arguments': '{"base_currency_value": 10}', 'name': 'currency_converter'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 113, 'total_tokens': 166, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BpMTwta8boW3aJKvNWn8uQKXOApay', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--870115e2-63aa-493f-a1fb-c610194f1ae4-0', tool_calls=[{'name': 'get_conversion_factor', 'args':

In [81]:
messages.append(AImessage)

In [82]:
(AImessage.tool_calls)

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'PKR'},
  'id': 'call_QyRgXfRPU4lGOgN5Of6haBEX',
  'type': 'tool_call'},
 {'name': 'currency_converter',
  'args': {'base_currency_value': 10},
  'id': 'call_K0D9xLIcfYpx8gHjtija1fc1',
  'type': 'tool_call'}]

In [83]:
import json
for tool_call in AImessage.tool_calls:
    if tool_call['name']=="get_conversion_factor":
        toolMessage1=get_conversion_factor.invoke(tool_call)
        conversion_rate=(json.loads(toolMessage1.content))['conversion_rate']
        messages.append(toolMessage1)
    if tool_call['name']=="currency_converter":
        tool_call['args']['conversion_rate']=conversion_rate
        toolMessage2=currency_converter.invoke(tool_call)
        messages.append(toolMessage2)
        

In [84]:
res=llm_with_tools.invoke(messages)
res.content

'The conversion factor between USD and PKR is 283.9313. \n\nWhen 10 USD is converted into PKR, it equals 2839.313 PKR.'